# 协程
- 一种微线程，用户态的上下文管理，性能开销小。
- 多个协程由同一个线程统一调度，如果一个协程没有发生异步IO，则会独占线程，直至任务执行结束。没有像线程一样的时间轮。

## async方法
async修饰的方法就是一个协程方法，返回一个协程对象(coroutine)

## 协程方法的使用
调用协程方法不会执行方法,需要通过asyncio.run(work())执行
### run做的事：
      1. 创建一个事件循环
      2. 将收到的携程对象，包装成一个task，交给事件循环
      3. 启动事件循环
      4. 阻塞当前线程，直到任务结束，返回结果
### 注意
- 在原生python文件中，需要通过`asyncio.run(main())`运行协程任务才能执行，但在jupyter中，因为已经全局启动了一个事件循环，因此可以通过`await main()`直接调用,如果还用`asyncio.run(main())`，相当于在已有活跃的事件循环中调用run，会出现异常：`RuntimeError: asyncio.run() cannot be called from a running event loop`

In [ ]:
import asyncio

async def work():
    print('haha1')
    print('haha2')
    print('haha3')
    return "meili"

print(await work())

# result = asyncio.run(work())
# print(result)

## await
 只有当await后面跟着异步IO任务时，会把当前任务挂起，会把CPU控制权交给事件循环，完成CPU的切换，最大化CPU利用率
 await只能在async方法中使用，后面只能接可等待对象。
### Common await obj:
    1. Coroutine
    2. Future
    3. Task

In [4]:
import asyncio

async def work():
    print('==work start')
    print('==work running...')
    # 模拟IO操作
    await asyncio.sleep(2)
    print('==work end')
    return "work result"

async def main():
    print('main start')
    print('main running...')
    res = await work()
    print(f'main get result: <{res}>')
    print('main end')
    return "main result"

# 在原生python文件中，需要通过asyncio.run()
# res = asyncio.run(main())
# print(res)

resp = await main()
print(f'main get result: <{resp}>')


main get result: <<coroutine object main at 0x000002635DF382E0>>


## 异步多任务执行
- `asyncio.creat_task(coroutine)`创建异步任务
    - 把协程对象包装为事件循环任务
    - 把任务注册到事件循环
    - 实现协程任务的并发执行，await再拿结果

In [1]:
import asyncio
import time

async def work(n):
    print(f'==work{n} start')
    print(f'==work{n} running...')
    # 模拟IO操作，切换CPU
    await asyncio.sleep(2)
    print(f'==work{n} end')
    return f"work{n} result"

async def main_sync():
    start = time.time()
    res1 = await work(1)
    print(res1)
    res2 = await work(2)
    print(res2)
    res3 = await work(3)
    print(res3)
    print(f'同步耗时{time.time() - start}秒')

async def main_async():
    start = time.time()
    task1 =  asyncio.create_task(work(1))
    task2 =  asyncio.create_task(work(2))
    task3 =  asyncio.create_task(work(2))
    # 阻塞等待task1结果
    res1 = await task1
    print(res1)
    res1 = await task2
    print(res1)
    res3 = await task3
    print(res3)
    print(f'异步耗时{time.time() - start}秒')

async def main_async_pro():
    """
    异步优化，适配批量创建多个协程任务
    :return:
    """
    start = time.time()
    g = (work(i) for i in range(3))
    print(f'p type is : {type(g)}')
    res_list = await asyncio.gather(*(work(i) for i in range(3)))
    print(res_list)
    print(f'异步耗时{time.time() - start}秒')


# 同步
# await main_sync()
# 异步
await main_async()
# 异步2
# await main_async_pro()

p type is : <class 'generator'>
==work0 start
==work0 running...
==work1 start
==work1 running...
==work2 start
==work2 running...
==work0 end
==work1 end
==work2 end
['work0 result', 'work1 result', 'work2 result']
异步耗时2.0045058727264404秒


## TODO t2await等待其余可等待对象的示例
